# 05 · Reading from GCS & BigQuery

> **Module 2 · Spark Fundamentals · Lecture 2.5**

In this notebook you will:

- Read CSV, JSON, and Parquet from `gs://` paths, glob patterns, and `recursiveFileLookup`
- Read BigQuery with the bundled connector: table reads, query reads, and incremental reads with a watermark
- Tune BigQuery reads with `materialization` and `maxReadParallelism`
- Configure the GCS connector and choose an error mode (`FAILFAST`, `PERMISSIVE`, `DROPMALFORMED`)
- Quarantine malformed records with `_corrupt_record`, and read JDBC tables and Pub/Sub streams

---


## Setup / prerequisites

- **Python 3.9–3.11** and **Java 11 or 17** on your machine.
- **PySpark 3.5.x**: `pip install "pyspark==3.5.1" pandas pyarrow numpy`
- This notebook is **self-contained and offline**: every code cell runs top-to-bottom with
  no GCP account. All data operations use local paths under `/tmp/`.
- GCS / BigQuery / Pub/Sub / JDBC code appears in **clearly-marked optional cells** that
  you uncomment after authenticating with `gcloud auth application-default login`.
- On Dataproc, **do not `pip install pyspark`** — use the Spark runtime already on the
  image (Spark 3.5 on Dataproc 2.2).


## 1 · The unified reader

Every read goes through `DataFrameReader` (`spark.read`). The shortcut methods are sugar
over the canonical form:

```python
spark.read.csv(path)  ==  spark.read.format("csv").load(path)
```

The GCS connector is **built into Dataproc** — no extra JAR. The path scheme decides the
filesystem: `gs://` works out of the box, `s3a://` and `hdfs://` do not.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pyspark

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("notebook-01")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("PySpark :", pyspark.__version__)
print("Spark   :", spark.version)
print("Master  :", spark.sparkContext.master)


In [ ]:
# ------------------------------------------------------------------
# Self-contained sample data that mirrors a GCS layout.
# ------------------------------------------------------------------
import json
import os
import shutil

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, TimestampType,
)

DATA_DIR = "/tmp/dataproc_notebook_05"
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)

# Mimic a partitioned lake: dt=YYYY-MM-DD subdirectories.
for day in ["2026-10-01", "2026-10-02"]:
    day_dir = os.path.join(DATA_DIR, "orders", f"dt={day}")
    os.makedirs(day_dir, exist_ok=True)
    with open(os.path.join(day_dir, "part-0000.csv"), "w", encoding="utf-8") as fh:
        fh.write("order_id,customer_id,amount,quantity\n")
        fh.write(f"o-{day}-1,c-1,100.50,2\n")
        fh.write(f"o-{day}-2,c-2,80.00,1\n")

# A JSON events directory with one deliberately broken line.
events_dir = os.path.join(DATA_DIR, "events")
os.makedirs(events_dir, exist_ok=True)
with open(os.path.join(events_dir, "events-0001.json"), "w", encoding="utf-8") as fh:
    fh.write(json.dumps({"event_id": "e-1", "event_type": "click", "amount": 10.0}) + "\n")
    fh.write('{"event_id": "e-2", "event_type": "purchase", "amount":}   # broken on purpose\n')
    fh.write(json.dumps({"event_id": "e-3", "event_type": "view", "amount": 0.0}) + "\n")

# Nested directory, to demonstrate recursiveFileLookup.
nested_dir = os.path.join(DATA_DIR, "nested", "level1", "level2")
os.makedirs(nested_dir, exist_ok=True)
(
    spark.createDataFrame(
        [("o-x", "c-1", 10.0, 1, "2026-10-01 10:00:00")],
        schema="order_id string, customer_id string, amount double, quantity int, ts string",
    )
    .withColumn("ts", F.col("ts").cast(TimestampType()))
    .write.mode("overwrite")
    .parquet(os.path.join(nested_dir, "deep"))
)

print("local layout mirrors a GCS lake:")
for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for f in files:
        print("  " * (level + 1) + f)

## 2 · Reading CSV / JSON / Parquet from a path

The same calls work against `gs://bucket/path/...` on Dataproc — only the path changes.

| Path form | Depth | Notes |
|---|---|---|
| `gs://bucket/orders/` | one level | Does **not** recurse; can pick up `_SUCCESS` |
| `gs://bucket/orders/*.csv` | one level | Glob over immediate children |
| `gs://bucket/orders/**/*.parquet` | any depth | `**` is the recursive wildcard |
| `.option("recursiveFileLookup","true")` | any depth | Also skips `_`/`.` hidden files |
| `.load(paths=[...])` | explicit | The only way to read a heterogeneous list of paths |

**Note:** **Prefer a narrow glob over a broad directory read.** `gs://bucket/dt=2026-10-01/*.parquet`
lists one partition; `gs://bucket/` lists the whole lake.

In [ ]:
orders_schema = StructType([
    StructField("order_id",    StringType(),    True),
    StructField("customer_id", StringType(),    True),
    StructField("amount",      DoubleType(),   True),
    StructField("quantity",    IntegerType(),  True),
])

# Local stand-in for gs://bucket/orders/dt=*/part-*.csv
#
# Read the BASE directory, not a leaf file. Spark applies partition discovery to
# any `key=value` ancestor of the path, so pointing it straight at
# `dt=2026-10-01/part-0000.csv` filters on a `dt` partition column that is not in
# the schema — and the read returns ZERO rows without raising an error.
csv_df = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .schema(orders_schema)
    .option("recursiveFileLookup", "true")
    .csv(f"{DATA_DIR}/orders")
)
csv_df.show(truncate=False)
print("csv rows:", csv_df.count())

# JSON
json_df = spark.read.json(f"{DATA_DIR}/events/events-0001.json")
json_df.show(truncate=False)

# Parquet
pq_df = spark.read.parquet(f"{DATA_DIR}/nested/level1/level2/deep")
pq_df.show(truncate=False)


### Multiple paths and recursive lookup

`load(paths=[...])` is the only way to read a heterogeneous list of paths. The
`recursiveFileLookup` option traverses subdirectories and skips `_SUCCESS` markers.

In [ ]:
# A GLOB narrows the read to one kind of file under the partition dirs.
# `orders/*/part-*.csv` matches every partition directory exactly one level down.
globbed = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .schema(orders_schema)
    .csv(f"{DATA_DIR}/orders/*/part-*.csv")
)
print("globbed rows:", globbed.count())
globbed.show(truncate=False)

# To read the partitions ONE AT A TIME and combine them, read each partition
# directory separately and union. This is the pattern you use when the
# partitions live in different buckets or need different schemas.
day1 = (
    spark.read
    .option("header", True)
    .schema(orders_schema)
    .csv(f"{DATA_DIR}/orders/dt=2026-10-01")
)
day2 = (
    spark.read
    .option("header", True)
    .schema(orders_schema)
    .csv(f"{DATA_DIR}/orders/dt=2026-10-02")
)
combined = day1.unionByName(day2)
print("\ncombined rows:", combined.count())
combined.show(truncate=False)

# recursiveFileLookup finds files at any depth (and skips hidden files).
recursive = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet(f"{DATA_DIR}/nested")
)
print("\nrecursive rows:", recursive.count())
recursive.show(truncate=False)


## 3 · Error modes

| Mode | Behaviour | Use when |
|---|---|---|
| `FAILFAST` | Abort the job on the first malformed record | Data must be perfect; you want to know immediately |
| `PERMISSIVE` | Malformed rows are dropped, the rest load | **CSV default** — never on financial data |
| `DROPMALFORMED` | Malformed rows are dropped, valid rows load | Clearer intent than PERMISSIVE |
| `EXCEPTION` | Renamed to `FAILFAST` in Spark 3.0 | Legacy name — silently means PERMISSIVE today |

**Caution:** CSV `PERMISSIVE` drops the **entire row**, not just the bad field, and gives no warning.

In [ ]:
bad_csv = f"{DATA_DIR}/bad.csv"
with open(bad_csv, "w", encoding="utf-8") as fh:
    fh.write("order_id,customer_id,amount,quantity\n")
    fh.write("o-1,c-1,100.50,2\n")
    fh.write("o-2,c-2,NOT_A_NUMBER,1\n")

permissive = (
    spark.read.option("header", True).option("mode", "PERMISSIVE")
    .option("inferSchema", True).csv(bad_csv)
)
print("PERMISSIVE rows:", permissive.count())   # the bad row vanished

try:
    failfast = (
        spark.read.option("header", True).option("mode", "FAILFAST")
        .option("inferSchema", True).csv(bad_csv)
    )
    failfast.count()
except Exception as exc:
    print("FAILFAST raised:", type(exc).__name__)
    print(str(exc).splitlines()[0][:120])

## 4 · Quarantine with `_corrupt_record`

JSON `PERMISSIVE` mode is smarter than CSV: the bad record is kept and its raw text is
placed in a `_corrupt_record` column.

**Note:** **Always quarantine, never silently drop.** A dropped row is a business incident you
will never see; a quarantined row is a ticket you can triage.

In [ ]:
# Quarantine the malformed rows instead of losing them.
#
# Spark 2.3+ refuses a query whose only referenced column is the internal
# `_corrupt_record` on a raw JSON/CSV source. CACHE first, then filter —
# the error message tells you to do exactly this.
events_raw = (
    spark.read
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .json(f"{DATA_DIR}/events/events-0001.json")
    .cache()          # <- required before filtering on _corrupt_record
)

good = events_raw.filter(F.col("_corrupt_record").isNull())
bad = events_raw.filter(F.col("_corrupt_record").isNotNull())

print("good rows:", good.count())
good.show(truncate=False)

print("\nquarantined rows:", bad.count())
bad.select("_corrupt_record").show(truncate=False)

# Write the bad rows to a dead-letter path for investigation.
bad.select("_corrupt_record").write.mode("overwrite").json(
    f"{DATA_DIR}/quarantine/events"
)
print("quarantine written to", f"{DATA_DIR}/quarantine/events")

events_raw.unpersist()


## 5 · GCS connector configuration

| Property | Default | What it controls |
|---|---|---|
| `spark.hadoop.fs.gs.impl` | set by the image | Binds Hadoop's `FileSystem` SPI to the GCS implementation |
| `fs.gs.connector.maximum-file-count` | `1000000` | Cap on files one scan may list |
| `fs.gs.connector.parallelism` | `64` | Threads used for parallel list/stat |
| `fs.gs.connector.maximum-buffer-size` | `104857600` (100 MB) | Write-ahead buffer for large objects |
| `fs.gs.connector.http.timeout` | `120000` ms | Timeout for GCS HTTP calls |

**Caution:** Raising `maximum-file-count` is not free — listing 10 million files takes real time
and real API quota. The better fix is a narrower glob or a partitioned layout.

In [ ]:
# These are the values Dataproc sets for you; print them for reference.
for key in ["spark.hadoop.fs.gs.impl",
            "fs.gs.connector.maximum-file-count",
            "fs.gs.connector.parallelism",
            "fs.gs.connector.maximum-buffer-size"]:
    try:
        print(f"{key:45s} = {spark.conf.get(key)}")
    except Exception:
        print(f"{key:45s} = <not set locally; provided by the Dataproc image>")

# Raising the file-count cap (only when you really need it).
spark.conf.set("fs.gs.connector.maximum-file-count", "10000000")
print("\nmaximum-file-count is now:", spark.conf.get("fs.gs.connector.maximum-file-count"))

## 6 · BigQuery reads — the three shapes

```python
# 1. A whole table
spark.read.format("bigquery") \
    .option("dataset", "my_project.sales").option("table", "orders").load()

# 2. A query
spark.read.format("bigquery") \
    .option("query", "SELECT order_id, amount FROM `my_project.sales.orders`").load()

# 3. By fully-qualified path
spark.read.format("bigquery").load("my_project.sales.orders")
```

**Caution:** The `query` option bills for the bytes the **query returns**. A `SELECT *` from a
1 TB partitioned table costs real money even if the filter matches 1%.

The BigQuery connector is bundled with Dataproc but versioned **separately** from Spark.
The artifact is `com.google.cloud.spark:bigquery-spark-sql_2.12:<version>` — the `_2.12`
suffix is the Scala binary version and must match.

### Optional: BigQuery reads (requires a GCP account)

Uncomment after `gcloud auth application-default login` and setting `BQ_PROJECT`.

In [ ]:
# ------------------------------------------------------------------
# OPTIONAL — requires GCP credentials. Uncomment to run.
# ------------------------------------------------------------------
# BQ_PROJECT = "your-project-id"
#
# # Shape 1: read a whole table
# bq_table = (
#     spark.read
#     .format("bigquery")
#     .option("dataset", f"{BQ_PROJECT}.sales")
#     .option("table", "orders")
#     .load()
# )
# bq_table.printSchema()
#
# # Shape 2: read via a query (project only the columns you need)
# bq_query = (
#     spark.read
#     .format("bigquery")
#     .option("query", f"""
#         SELECT order_id, customer_id, amount, order_date
#         FROM `{BQ_PROJECT}.sales.orders`
#         WHERE order_date >= '2026-10-01'
#     """)
#     .option("materialization", "true")
#     .option("materialization.project", f"{BQ_PROJECT}_tmp")
#     .option("materialization.dataset", "bq_materialization")
#     .option("maxReadParallelism", 8)
#     .load()
# )
# bq_query.show(10, truncate=False)

print("BigQuery read cells are commented out — no GCP account required for this notebook.")

### Incremental BigQuery reads

The connector supports a watermark-based read:

```python
spark.read.format("bigquery") \
    .option("readAll", False) \
    .option("watermark", "2026-10-28T00:00:00Z") \
    .option("partitionField", "order_date") \
    .option("partitionExpirationMs", "86400000") \
    .load("my_project.sales.orders")
```

The simplest and most robust incremental read is an explicit **overlapping UTC range**
pushed into the query. Overlap plus idempotent downstream writes gives at-least-once
semantics without data loss.

**Note:** `materialization` (default `true` for query reads) materialises the query result into a
temporary table so it can be read in parallel. `maxReadParallelism` (default **1**) is
the number of parallel read streams — raising it to 8–16 routinely cuts read time by an
order of magnitude. Materialisation is **not** supported for direct table reads.

In [ ]:
# ------------------------------------------------------------------
# OPTIONAL — incremental BigQuery read. Uncomment to run.
# ------------------------------------------------------------------
# from datetime import datetime, timedelta, timezone
#
# def read_incremental(spark, project, dataset, table, days_back=1, parallelism=8):
#     end = datetime.now(timezone.utc)
#     start = end - timedelta(days=days_back)
#     query = f"""
#         SELECT * FROM `{project}.{dataset}.{table}`
#         WHERE updated_at >= TIMESTAMP('{start.isoformat()}')
#           AND updated_at <  TIMESTAMP('{end.isoformat()}')
#     """
#     return (
#         spark.read
#         .format("bigquery")
#         .option("query", query)
#         .option("materialization", "true")
#         .option("materialization.project", f"{project}_tmp")
#         .option("maxReadParallelism", parallelism)
#         .load()
#     )
#
# incremental = read_incremental(spark, BQ_PROJECT, "sales", "orders")
# incremental.show(10, truncate=False)

print("Incremental BigQuery read is available in the commented cell above.")

## 7 · JDBC reads

| Option | Effect |
|---|---|
| `url` | `jdbc:<driver>://host:port/database` |
| `dbtable` | A table name **or** a subquery in parentheses |
| `query` | A full SQL query — **cannot** be combined with `partitionColumn` |
| `partitionColumn` | Numeric/timestamp/date column to split on (monotonically increasing) |
| `lowerBound` / `upperBound` | The range to cover — **not** the min/max of the data |
| `numPartitions` | Number of parallel JDBC connections |
| `fetchsize` | Rows per network round trip — `10000` is a sane default |

**Note:** `dbtable` + `partitionColumn` pushes predicates down and lets the database use an
index. `query` re-runs your query once per partition.

**Caution:** Never hardcode passwords — use the cluster's service account or Secret Manager.

In [ ]:
# ------------------------------------------------------------------
# OPTIONAL — JDBC read. Uncomment to run against a real database.
# ------------------------------------------------------------------
# jdbc_df = (
#     spark.read
#     .format("jdbc")
#     .option("url", "jdbc:postgresql://db-host:5432/mydb")
#     .option("dbtable", "public.orders")
#     .option("user", "readonly")
#     .option("password", "secret")
#     .option("fetchsize", 10000)
#     .load()
# )
#
# jdbc_partitioned = (
#     spark.read
#     .format("jdbc")
#     .option("url", "jdbc:postgresql://db-host:5432/mydb")
#     .option("dbtable", "public.orders")
#     .option("partitionColumn", "order_id")
#     .option("lowerBound", "1")
#     .option("upperBound", "1000000")
#     .option("numPartitions", 10)
#     .option("fetchsize", 10000)
#     .load()
# )

print("JDBC read cells are commented out — no database required for this notebook.")

## 8 · Pub/Sub `readStream`

Pub/Sub is a **streaming** source, so it uses `spark.readStream`, not `spark.read`.

| Source | Key concepts |
|---|---|
| **Pub/Sub** | Uses the *subscription* model — the subscription tracks a cursor. `maxFilesPerTrigger` bounds each batch. |
| **Kafka** | Uses *offsets* and consumer groups. `startingOffsets` applies only on the first run. |

**Caution:** Both need a **checkpoint location** for exactly-once-ish semantics. Without one, a
restart reprocesses everything and you get duplicates.

In [ ]:
# ------------------------------------------------------------------
# OPTIONAL — Pub/Sub streaming read. Uncomment to run.
# ------------------------------------------------------------------
# stream = (
#     spark.readStream
#     .format("pubsub")
#     .option("projectId", "your-project-id")
#     .option("subscription", "orders-sub")
#     .option("maxFilesPerTrigger", 1000)
#     .load()
# )
#
# query = (
#     stream.writeStream
#     .format("parquet")
#     .option("checkpointLocation", "gs://your-bucket/checkpoints/orders_pubsub")
#     .option("path", "gs://your-bucket/stream/orders")
#     .trigger(processingTime="2 minutes")
#     .start()
# )
# query.awaitTermination()

print("Pub/Sub readStream cell is commented out — no GCP account required for this notebook.")

## 🔑 Key Takeaways

- Every read goes through `spark.read`; the GCS connector is built into Dataproc — no extra JAR.
- **Prefer a narrow glob over a broad directory read** — it is the difference between milliseconds and minutes.
- `recursiveFileLookup=true` traverses subdirectories and skips `_SUCCESS`/hidden files; `load(paths=[...])` handles heterogeneous path lists.
- Error modes are `FAILFAST`, `PERMISSIVE`, `DROPMALFORMED`. `EXCEPTION` was renamed in Spark 3.0 and silently means PERMISSIVE today.
- CSV `PERMISSIVE` drops the **whole row**; JSON `PERMISSIVE` keeps it in `_corrupt_record` — **quarantine, never silently drop**.
- BigQuery connector: table reads, query reads, and path loads. The `query` option bills for the bytes the query returns.
- `materialization` (query reads only) creates a temporary table so parallel reads are possible; `maxReadParallelism` (default 1) is a major performance lever.
- Incremental reads: use an overlapping UTC range in the query plus idempotent writes.
- JDBC: `dbtable` + `partitionColumn` pushes predicates down; `query` and `partitionColumn` are mutually exclusive.
- Pub/Sub and Kafka use `readStream` and require a checkpoint location.

---

## 📚 Next steps

- Run the BigQuery read cell against your own project and compare `maxReadParallelism` of 1 and 8.
- Continue to `06-writing-to-multiple-sinks.ipynb` for the write side.
